In [ ]:
# Embedding Generation Script (for Colab)
# Run this in Google Colab to precompute embeddings and download them for later use.

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from google.colab import files
import os

# ---------------- Configuration ----------------
DATA_PATH = '/content/cleaned_flipkart_data.csv'  # upload this CSV first
OUTPUT_PATH = '/content/flipkart_embeddings.npy'
MODEL_NAME = 'all-MiniLM-L6-v2'
TEXT_COL = 'search_text'
CATEGORY_COL = 'category'
MIN_CATEGORY_COUNT = 15

# ---------------- Load Data ----------------
print('Reading dataset...')
df = pd.read_csv(DATA_PATH)
df = df.rename(columns={c: c.strip() for c in df.columns})

if 'title' in df.columns:
    titles = df['title'].fillna('')
elif 'name' in df.columns:
    titles = df['name'].fillna('')
else:
    titles = df.iloc[:, 0].astype(str)

desc = df['description'].fillna('') if 'description' in df.columns else ''
df[TEXT_COL] = (titles + ' - ' + desc).str.strip()

if CATEGORY_COL not in df.columns:
    for col in ['category', 'cat', 'primary_category']:
        if col in df.columns:
            df[CATEGORY_COL] = df[col]
            break
    else:
        df[CATEGORY_COL] = 'unknown'

# keep only categories with enough samples
cat_counts = df[CATEGORY_COL].value_counts()
valid_cats = cat_counts[cat_counts >= MIN_CATEGORY_COUNT].index.tolist()
df = df[df[CATEGORY_COL].isin(valid_cats)].reset_index(drop=True)

print(f'Total valid products: {len(df)}')

# ---------------- Generate Embeddings ----------------
print('Loading embedding model...')
model = SentenceTransformer(MODEL_NAME)
texts = df[TEXT_COL].astype(str).tolist()
print(f'Encoding {len(texts)} items...')
embeds = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)
embeds = np.array(embeds, dtype=np.float32)

# ---------------- Save and Download ----------------
np.save(OUTPUT_PATH, embeds)
print(f'Embeddings saved to {OUTPUT_PATH}')

files.download(OUTPUT_PATH)
print('Download started for embeddings file.')

Reading dataset...
Total valid products: 20000
Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Embeddings saved to /content/flipkart_embeddings.npy


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started for embeddings file.


In [ ]:
# Load the CSV file and display its shape
csv_embeds_df = pd.read_csv(OUTPUT_PATH_CSV)
print(f"Shape of the embeddings CSV file: {csv_embeds_df.shape}")

Shape of the embeddings CSV file: (20000, 384)
